# 🧠 โปรเจกต์ NN: Data Preprocessing (การเตรียมข้อมูลสำหรับ Neural Network)
**Dataset:** Sleep Health and Lifestyle Dataset
**เป้าหมาย:** ทำความสะอาดและแปลงข้อมูลพฤติกรรมการนอนและสุขภาพ เพื่อใช้ทำนายความเสี่ยงโรคการนอนหลับ (Classification: None, Insomnia, Sleep Apnea)

## 1) Import Libraries (นำเข้าเครื่องมือที่จำเป็น)
โหลดไลบรารีพื้นฐานสำหรับการจัดการตารางข้อมูล (Pandas) และการบันทึกไฟล์ (Joblib)

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("✅ Import Libraries เรียบร้อยพร้อมลุย!")

✅ Import Libraries เรียบร้อยพร้อมลุย!


## 2) โหลดข้อมูลดิบ (Load Raw Data)
นำเข้าข้อมูล `Sleep_health_and_lifestyle_dataset.csv` และตรวจสอบโครงสร้าง

In [2]:
# ถอยหลัง 1 โฟลเดอร์เพื่อเข้าสู่โฟลเดอร์ data ของฝั่ง NN
df = pd.read_csv('../data/Sleep_health_and_lifestyle_dataset.csv')
print(f"ขนาดข้อมูลดิบ: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
display(df.head(3))

ขนาดข้อมูลดิบ: 374 แถว, 13 คอลัมน์


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN


## 3) การทำ Feature Engineering และ Data Cleaning
จุดนี้สำคัญมากสำหรับ Dataset ชุดนี้ (แก้ปัญหาความไม่สมบูรณ์ตามข้อกำหนด 1.c):
1. **ลบคอลัมน์ที่ไม่จำเป็น:** `Person ID` ไม่มีผลต่อการทำนาย
2. **Missing Values ที่มีความหมาย:** คอลัมน์ `Sleep Disorder` ที่เป็นค่าว่าง (NaN) หมายถึงบุคคลนั้น "ไม่มีโรค" เราจึงต้องเติมคำว่า `"None"` ลงไปแทน
3. **Inconsistent Labels:** คอลัมน์ `BMI Category` มีคำว่า `"Normal"` และ `"Normal Weight"` ซึ่งคือสิ่งเดียวกัน ต้องจับมารวมกัน
4. **String Parsing:** แยกคอลัมน์ `Blood Pressure` ("120/80") ออกเป็นความดันตัวบน (Systolic) และตัวล่าง (Diastolic) เพื่อให้เป็นตัวเลขที่โมเดลคำนวณได้

In [3]:
# 1. ลบคอลัมน์ Person ID
if 'Person ID' in df.columns:
    df = df.drop(columns=['Person ID'])

# 2. เติมค่าว่างใน Sleep Disorder ด้วยคำว่า 'None'
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
print(f"📊 สัดส่วนของคลาสเป้าหมาย (Target):\n{df['Sleep Disorder'].value_counts()}\n")

# 3. รวมคลาสใน BMI Category ให้เป็นมาตรฐานเดียวกัน
df['BMI Category'] = df['BMI Category'].replace('Normal Weight', 'Normal')

# 4. แยกความดันโลหิต (Blood Pressure) เป็น 2 คอลัมน์
if 'Blood Pressure' in df.columns:
    # แยกตัวเลขด้วยเครื่องหมาย '/'
    df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True)
    
    # แปลงชนิดข้อมูลให้เป็นตัวเลข (Integer)
    df['Systolic_BP'] = df['Systolic_BP'].astype(int)
    df['Diastolic_BP'] = df['Diastolic_BP'].astype(int)
    
    # ลบคอลัมน์เดิมทิ้ง
    df = df.drop(columns=['Blood Pressure'])
    print("✅ แยกคอลัมน์ Blood Pressure เป็น Systolic และ Diastolic เรียบร้อย")

display(df.head(3))

📊 สัดส่วนของคลาสเป้าหมาย (Target):
Sleep Disorder
None           219
Sleep Apnea     78
Insomnia        77
Name: count, dtype: int64

✅ แยกคอลัมน์ Blood Pressure เป็น Systolic และ Diastolic เรียบร้อย


,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic_BP,Diastolic_BP
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,None,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,None,125,80
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,None,125,80


## 4) การแปลงข้อความเป็นตัวเลข (Encoding)
- **Features:** แปลงคอลัมน์ `Gender`, `Occupation`, `BMI Category` ด้วย **One-Hot Encoding**
- **Target:** แปลงโรคการนอนหลับ `Sleep Disorder` ให้เป็นตัวเลข (0, 1, 2) ด้วย **LabelEncoder** (พร้อมเซฟตัวแปลงเก็บไว้ เพื่อใช้แปลงกลับเป็นข้อความตอนแสดงผลบนหน้าเว็บ)

In [4]:
# --- จัดการตัวแปรต้น (Features) ---
categorical_cols = ['Gender', 'Occupation', 'BMI Category']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

# --- จัดการตัวแปรตาม (Target) ---
# สร้างตัวแปลง LabelEncoder
le = LabelEncoder()
df_encoded['Sleep Disorder'] = le.fit_transform(df_encoded['Sleep Disorder'])

# สร้างโฟลเดอร์ models ถ้ายังไม่มี
os.makedirs('../models', exist_ok=True)

# บันทึก LabelEncoder เอาไว้ใช้แปลผลบนหน้าเว็บ (เช่น แปล 1 กลับเป็น Insomnia)
joblib.dump(le, '../models/nn_label_encoder.pkl')
print("💾 บันทึก 'nn_label_encoder.pkl' เรียบร้อย! (คลาสคือ: {})".format(le.classes_))

💾 บันทึก 'nn_label_encoder.pkl' เรียบร้อย! (คลาสคือ: ['Insomnia' 'None' 'Sleep Apnea'])


## 5) บันทึกข้อมูลและโครงสร้าง Features (Export Data & Features)
เก็บข้อมูลที่ Clean แล้ว และบันทึกรายชื่อคอลัมน์ เพื่อความพร้อมในการออกแบบเลเยอร์ของ Neural Network ในขั้นตอนต่อไป

In [5]:
# แยก Target ออกเพื่อดึงแค่ชื่อ Features
X = df_encoded.drop(columns=['Sleep Disorder'])
feature_names = X.columns.tolist()

# บันทึกรายชื่อคอลัมน์เป็นไฟล์ .pkl
joblib.dump(feature_names, '../models/nn_features.pkl')
print("💾 บันทึกโครงสร้าง Features ลง '../models/nn_features.pkl' เรียบร้อย")

# บันทึก Dataset ที่ Clean แล้ว
df_encoded.to_csv('../data/sleep_health_cleaned.csv', index=False)
print("💾 บันทึกข้อมูลพร้อมเทรนลง '../data/sleep_health_cleaned.csv' เรียบร้อย")

print("\n🎉 กระบวนการทำความสะอาดข้อมูลของ NN เสร็จสมบูรณ์!")

💾 บันทึกโครงสร้าง Features ลง '../models/nn_features.pkl' เรียบร้อย
💾 บันทึกข้อมูลพร้อมเทรนลง '../data/sleep_health_cleaned.csv' เรียบร้อย

🎉 กระบวนการทำความสะอาดข้อมูลของ NN เสร็จสมบูรณ์!
